# WO-03 Q4 — river names: is the Natural Earth layer useful here
Natural Earth 5.0.0 `ne_10m_rivers_lake_centerlines_scale_rank` (`data/`, read from disk; not in the database) has 4,224 lines, with visible gaps in deserts and mountains. This notebook asks whether it names the main stem near a place often enough, and reliably enough, to be worth using; the alternative is generic wording with no name. Place sets: the 258 World Heritage Cities (likely nearer major rivers than settlements in general, so an upper bound) and the 306 WO-02 random basin points (a lower bound). For each place: the nearest reach in `gaz.rivers` above an upstream-area threshold (Q5 query), the point on it closest to the place, and the nearest named Natural Earth river line to that point. Only `River` and `River (Intermittent)` lines with a name are used. Read-only. Dateline wrap is not handled.

In [1]:
%matplotlib inline
# Cell 1 — Setup: the shapefile (named river lines only), WH Cities, the sample points
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import Point
from shapely.ops import transform
from pyproj import CRS, Transformer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C

conn = db_connect()
conn.execute("SET statement_timeout = 120000")
SHP = ROOT / 'data' / 'ne_10m_rivers_lake_centerlines_scale_rank' / 'ne_10m_rivers_lake_centerlines_scale_rank.shp'
G = gpd.read_file(SHP)
G['label_name'] = G['name_en'].where(G['name_en'].notna(), G['name'])
is_river = G['featurecla'].isin(['River', 'River (Intermittent)'])
NE = G[is_river & G['label_name'].notna()].reset_index(drop=True)
NE_SINDEX = NE.geometry.sindex
CITIES = pd.DataFrame(conn.execute("SELECT id, city, country, region, ST_Y(geom::geometry), ST_X(geom::geometry) FROM gaz.wh_cities ORDER BY id").fetchall(),
                      columns=['id', 'city', 'country', 'region', 'lat', 'lon'])
SAMPLE = C.population_points()
print('commit', C.commit(), '| shapefile lines:', len(G), '| used (river class with a name):', len(NE), '| distinct names:', NE['label_name'].nunique())
print('excluded: lake centerlines', int((G['featurecla'] == 'Lake Centerline').sum()), '| canals', int((G['featurecla'] == 'Canal').sum()),
      '| river class without a name', int((is_river & G['label_name'].isna()).sum()))
print('used lines by scalerank:', NE['scalerank'].value_counts().sort_index().to_dict())
print('WH Cities:', len(CITIES), '| sample points:', len(SAMPLE))

commit 23d4144 | shapefile lines: 4224 | used (river class with a name): 3589 | distinct names: 1044
excluded: lake centerlines 479 | canals 6 | river class without a name 150
used lines by scalerank: {1: 95, 2: 114, 3: 235, 4: 238, 5: 200, 6: 580, 7: 799, 8: 830, 9: 460, 10: 37, 12: 1}
WH Cities: 258 | sample points: 306


In [2]:
# Cell 2 — Query functions; nearest named line (metric distance) and the nearest reach above a threshold
THRESHOLDS = [1000, 10000]
RADII = [0.25, 0.5, 1, 2, 4, 8, 16, 32, 64]

def ne_nearest(lat, lon, n_keep=2):
    """Distinct-name nearest NE river lines to a point: [(name, km, scalerank), ...] up to n_keep, nearest first."""
    pt = Point(lon, lat)
    tr = Transformer.from_crs(CRS.from_epsg(4326), CRS.from_proj4(f"+proj=aeqd +lat_0={lat} +lon_0={lon} +datum=WGS84 +units=m"), always_xy=True)
    inscribed = lambda r: r * 111.0 * max(np.cos(np.radians(abs(lat))), 0.05)
    best = []
    for r in RADII:
        ids = NE_SINDEX.query(pt.buffer(r))
        if len(ids) == 0:
            continue
        rows = []
        for i in ids:
            g = transform(tr.transform, NE.geometry.iloc[i])
            rows.append((NE['label_name'].iloc[i], g.distance(Point(0, 0)) / 1000.0, int(NE['scalerank'].iloc[i])))
        rows.sort(key=lambda x: x[1])
        best = rows
        if rows[0][1] <= inscribed(r):
            break
    out, seen = [], set()
    for nm, km, sr in best:
        if nm not in seen:
            seen.add(nm); out.append((nm, km, sr))
        if len(out) == n_keep:
            break
    return out

STEM_SQL = """
WITH p AS (SELECT ST_SetSRID(ST_MakePoint(%s, %s), 4326) AS geom)
SELECT r.hyriv_id::bigint, r.upland_skm::float8, ST_Distance(r.geog, p.geom::geography) / 1000.0,
       ST_X(ST_ClosestPoint(r.geom, p.geom)), ST_Y(ST_ClosestPoint(r.geom, p.geom))
FROM gaz.rivers r, p WHERE r.upland_skm >= %s ORDER BY r.geog <-> p.geom::geography LIMIT 1
"""

def place_row(setname, pid, label, group, lat, lon):
    rows = []
    near = ne_nearest(lat, lon, 1)
    for thr in THRESHOLDS:
        s = conn.execute(STEM_SQL, (lon, lat, thr)).fetchone()
        nn = ne_nearest(s[4], s[3], 2)
        d = dict(set=setname, id=pid, label=label, group=group, lat=lat, lon=lon, threshold=thr,
                 stem_reach=s[0], stem_up_km2=s[1], stem_km=s[2],
                 place_named_name=near[0][0], place_named_km=near[0][1],
                 stem_named_name=nn[0][0], stem_named_km=nn[0][1], stem_named_rank=nn[0][2],
                 second_name=nn[1][0] if len(nn) > 1 else '', second_km=nn[1][1] if len(nn) > 1 else np.nan)
        rows.append(d)
    return rows

t_start, rows = time.perf_counter(), []
for k, r in enumerate(CITIES.itertuples(), 1):
    rows += place_row('wh_cities', int(r.id), f'{r.city}, {r.country}', r.region, float(r.lat), float(r.lon))
    if k % 50 == 0:
        print(f'  cities {k} of {len(CITIES)} in {time.perf_counter() - t_start:.0f}s')
for k, s in enumerate(SAMPLE, 1):
    rows += place_row('sample', int(s['hybas_id_l6']), str(s['hybas_id_l6']), s['climate_zone'], float(s['lat']), float(s['lon']))
    if k % 100 == 0:
        print(f'  sample {k} of {len(SAMPLE)} in {time.perf_counter() - t_start:.0f}s')
R4 = pd.DataFrame(rows)
R4.to_csv(C.OUT / 'q4_names.csv', index=False)
print(f'rows saved: {len(R4)} | this run took {time.perf_counter() - t_start:.0f}s')

  sample 300 of 306 in 21s
rows saved: 1128 | this run took 21s


In [3]:
# Cell 3 — How often is there a named river line near the main stem?
L = []
for setname in ['wh_cities', 'sample']:
    for thr in THRESHOLDS:
        d = R4[(R4['set'] == setname) & (R4['threshold'] == thr)]
        n = len(d)
        cuts = {km: int((d['stem_named_km'] <= km).sum()) for km in (5, 10, 25, 50)}
        L.append(f"{setname}, stem above {thr} km2 (n={n}): main stem to a named line within " +
                 ' | '.join(f'{km} km {v} ({100 * v / n:.0f}%)' for km, v in cuts.items()))
        L.append(f"   distance place to its nearest reach, km: median {d['stem_km'].median():.1f} | 90% {d['stem_km'].quantile(0.9):.1f}; "
                 f"place to nearest named line (any river), km: median {d['place_named_km'].median():.1f} | 90% {d['place_named_km'].quantile(0.9):.1f}")
L.append('')
L.append('WH Cities, stem above 10,000 km2: named line within 10 km of the main stem, by region')
d = R4[(R4['set'] == 'wh_cities') & (R4['threshold'] == 10000)]
by = d.groupby('group')['stem_named_km'].agg(n='count', within_10=lambda s: int((s <= 10).sum()), within_50=lambda s: int((s <= 50).sum()))
L.append(by.to_string())
L.append('')
L.append('Sample, stem above 10,000 km2: named line within 10 km, by climate zone')
d = R4[(R4['set'] == 'sample') & (R4['threshold'] == 10000)]
by = d.groupby('group')['stem_named_km'].agg(n='count', within_10=lambda s: int((s <= 10).sum()), within_50=lambda s: int((s <= 50).sum()))
L.append(by.to_string())
print('\n'.join(L))

wh_cities, stem above 1000 km2 (n=258): main stem to a named line within 5 km 71 (28%) | 10 km 74 (29%) | 25 km 88 (34%) | 50 km 119 (46%)
   distance place to its nearest reach, km: median 8.4 | 90% 41.7; place to nearest named line (any river), km: median 59.3 | 90% 453.4
wh_cities, stem above 10000 km2 (n=258): main stem to a named line within 5 km 153 (59%) | 10 km 156 (60%) | 25 km 168 (65%) | 50 km 179 (69%)
   distance place to its nearest reach, km: median 59.2 | 90% 201.0; place to nearest named line (any river), km: median 59.3 | 90% 453.4
sample, stem above 1000 km2 (n=306): main stem to a named line within 5 km 50 (16%) | 10 km 56 (18%) | 25 km 74 (24%) | 50 km 103 (34%)
   distance place to its nearest reach, km: median 9.0 | 90% 30.3; place to nearest named line (any river), km: median 108.9 | 90% 1336.1
sample, stem above 10000 km2 (n=306): main stem to a named line within 5 km 109 (36%) | 10 km 115 (38%) | 25 km 120 (39%) | 50 km 131 (43%)
   distance place to its neare

In [4]:
# Cell 4 — Spot check: 30 random WH Cities (stem above 10,000 km2), with the second-nearest differently named line
d = R4[(R4['set'] == 'wh_cities') & (R4['threshold'] == 10000)].sample(30, random_state=7).sort_values('label')
show = d[['label', 'stem_up_km2', 'stem_km', 'stem_named_name', 'stem_named_km', 'second_name', 'second_km']].copy()
for c in ['stem_up_km2', 'stem_km', 'stem_named_km', 'second_km']:
    show[c] = show[c].round(1)
print(show.to_string(index=False))
amb = R4[(R4['set'] == 'wh_cities') & (R4['threshold'] == 10000) & (R4['stem_named_km'] <= 25) & (R4['second_km'] <= 25)]
print(f"WH Cities with two differently named lines within 25 km of the main stem point: {len(amb)} of {int((R4['set'] == 'wh_cities').sum() / len(THRESHOLDS))}")

                        label  stem_up_km2  stem_km stem_named_name  stem_named_km second_name  second_km
       Amsterdam, Netherlands     160172.3     44.3             Lek            0.3        Waal       16.8
            Augsburg, Germany      12035.8     33.6          Danube            1.2                    NaN
              Berlin, Germany      12752.3     11.7            Oder           71.9        Elbe       81.2
             Brasília, Brazil      10593.5    105.2       Tocantins           66.5     Corumbá       77.5
              Bruges, Belgium      10253.3     43.4         Scheldt            0.7                    NaN
           Cuernavaca, Mexico      22384.7    101.3          Balsas            0.4                    NaN
                  Cusco, Peru      14185.4     27.0         Ucayali            1.4                    NaN
               Córdoba, Spain      24858.9      1.6    Guadalquivir            0.8                    NaN
               Dakar, Senegal      32922.3    